# IV. backprop, by hand

*Building the backward pass of a full MLP — matrix multiply, `tanh`, batch normalization, and cross-entropy — one gradient at a time, then collapsing the ugly parts into single lines.*

This is part 4 of [makemore](https://github.com/karpathy/makemore), following the [lecture](https://www.youtube.com/watch?v=q8SA3rM6ckI) by Andrej Karpathy. In [part 3]({{ '/projects/makemore_batchnorm/' | relative_url }}) I added batch normalization and watched it decouple the health of the network from the scale of its initialization. That project treated `loss.backward()` as a black box. This one opens the box.

The exercise is to backpropagate through the entire network **by hand** — no autograd — computing the gradient of the loss with respect to every intermediate tensor and every parameter, and checking each one against PyTorch's own `.grad` to the last decimal. Then, having done it the tedious way, to find the places where an entire block of the backward pass collapses into a single expression: the cross-entropy gradient becomes `probs - onehot`, and the batch-norm gradient becomes one line that never mentions the seven intermediate tensors it flows through.

I want to be honest about why this is worth doing, because the payoff is not the formulas — those are in every textbook. The payoff is a *method*. By the end I had one principle that generated every gradient in the network without guessing:

> **Think in single scalars. Trace the routes an input takes to the loss. Sum over them. Then read the index pattern back as a matrix operation.**

Everything below is that principle applied over and over, in increasing difficulty, until it handles the hardest node — the batch-norm mean-and-variance coupling — without any new ideas. What follows traces the real path I took, including the walls I hit, because the walls are where the understanding actually lives.


## Setup

The scaffolding is inherited from part 3: read the names, build the character vocabulary, cut the dataset into train/dev/test, and initialize a one-hidden-layer MLP with a batch-norm layer. Nothing here is new — the interesting work starts once the forward pass is defined.


In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
names = open('names.txt', 'r').read().splitlines()
print(f"total names in dataset: {len(names)}")

In [ ]:
# build character vocabulary and mapping
chars = sorted(list(set(''.join(names))))
ctoi = {c:i+1 for i,c in enumerate(chars)}
ctoi['.'] = 0
itoc = {i:c for c, i in ctoi.items()}
vocab_size = len(itoc)
print(f"characters: {vocab_size}")

In [ ]:
# build the dataset
block_size = 3 # context length

def build_dataset(names):
    X, Y = [], []

    for n in names:
        context = [0] * block_size
        for ch in n + '.':
            ix = ctoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(1994)
random.shuffle(names)
n1 = int(0.8*len(names))
n2 = int(0.9*len(names))

print("Xtrn, Ytrn = 80%")
Xtrn, Ytrn = build_dataset(names[:n1])
print("Xdev, Ydev = 10%")
Xdev, Ydev = build_dataset(names[n1:n2])
print("Xtst, Ytst = 10%")
Xtst, Ytst = build_dataset(names[n2:])

### The gradient checker

The whole project rests on one utility. `cmp` compares a hand-computed gradient `dt` against PyTorch's `.grad` for the same tensor, three ways:

- **exact** — are the two arrays bit-for-bit identical?
- **approximate** — do they agree within floating-point tolerance (`torch.allclose`)?
- **maxdiff** — the largest absolute disagreement anywhere.

A note that saves a great deal of panic later: **`exact: True` is a bonus, not the target.** Floating-point addition is not associative, so a hand-derived expression and autograd's internal kernel can compute the mathematically identical gradient in a different operation order and land a few ULPs apart. When that happens `exact` reads `False` while `approximate` reads `True` and `maxdiff` sits around `1e-9`. That is a *pass*. The real success criterion throughout is `approximate: True` with a tiny `maxdiff`.


In [ ]:
# utility function to compare manually- to pytorch-computed gradients
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

### Initialization

Standard Kaiming-flavored initialization for the weights (the `5/3` gain compensates for `tanh`, as derived in part 3), plus the batch-norm scale `bn_gain` (γ) and shift `bn_bias` (β). Every parameter gets `requires_grad = True` so PyTorch will compute the reference gradients we check against.


In [ ]:
# initializing the model
n_embedding = 10 # size of the embedding dimension
n_hidden = 64 # number of neurons in the hidden layer
fan_in = n_embedding * block_size
g = torch.Generator().manual_seed(1994) # generator for reproducibility

C = torch.randn((vocab_size, n_embedding),            generator=g)
# layer 1
W1 = torch.randn((fan_in, n_hidden),  generator=g) * (5/3)/(fan_in**0.5)
b1 = torch.randn(n_hidden,                            generator=g) * 0.1
# layer 2
W2 = torch.randn((n_hidden, vocab_size),              generator=g) * 0.1
b2 = torch.randn(vocab_size,                          generator=g) * 0.1
# batch norm parameters
bn_gain = torch.randn((1, n_hidden)) * 0.1 + 1.0
bn_bias = torch.randn((1, n_hidden)) * 0.1

parameters = [C, W1, b1, W2, b2, bn_bias, bn_gain]

for p in parameters:
    p.requires_grad = True
    
print(f"number of parameters = {sum(p.nelement() for p in parameters)}")

In [ ]:
# construct a minibatch
batch_size = 32
n = batch_size
ix = torch.randint(0, Xtrn.shape[0], (n,), generator=g)
Xb, Yb = Xtrn[ix], Ytrn[ix]

## The forward pass, fully decomposed

To backpropagate by hand, the forward pass has to be broken into atomic steps — every operation gets its own named intermediate tensor, so that each has a `.grad` to check against. This is deliberately more verbose than one would ever write in practice; `bn_del`, `bn_del2`, `counts_sum_inv`, and the rest exist only so that the chain rule has somewhere to land at each step.

The network computes, in order:

$$
\text{emb} \;\to\; z\_pre = \text{emb}\,W_1 + b_1 \;\to\; \underbrace{\hat z = \frac{z\_pre - \mu}{\sqrt{\sigma^2 + \epsilon}}, \quad z = \gamma\,\hat z + \beta}_{\text{batch norm}} \;\to\; h = \tanh(z) \;\to\; \text{logits} = h\,W_2 + b_2 \;\to\; \text{loss}
$$

with the loss being cross-entropy, itself decomposed into the max-subtraction (for numerical stability), exponentiation, normalization, log, and mean. The batch statistics use Bessel's correction — variance divides by $n-1$, not $n$ — which matters enormously once we differentiate them.

The cell also calls `loss.backward()` and retains the gradient on every intermediate, giving us the reference gradients for the entire notebook.


In [ ]:
# fully-specified forward pass

# embed training inputs [32, 30]
emb_uncat = C[Xb]
emb = emb_uncat.view(emb_uncat.shape[0], -1) # flattens Xb context window index dimension

# linear layer 1 [32, 64]
z_pre = emb @ W1 + b1 # hidden layer pre-activation

# batch normalization layer 1 [32, 64]
bn_mean = 1/n*z_pre.sum(0, keepdim=True) # sum of each row divided by n: [64, 1]
bn_del = z_pre - bn_mean # delta between pre-activations and row mean: [32, 64]
bn_del2 = bn_del**2
bn_var = 1/(n-1)*(bn_del2).sum(0, keepdim=True) # var = sig^2 = del^2/n, with Bessel's correction /(n-1) instead of n
bn_var_inv = (bn_var + 1e-5)**-0.5 # small scalar to avoid /0
bn_raw = bn_del * bn_var_inv # intermediate term for cleanliness
z = bn_gain * bn_raw + bn_bias # pre-activation after batch normalization

# non-linear layer [32, 64]
h = torch.tanh(z) # hidden layer activation

# linear layer 2 [32, 27]
logits = h @ W2 + b2 # output layer

# cross entropy loss (equivalent to F.cross_entropy(logits, Yb)
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # ensures numerical stability
counts = norm_logits.exp() # [32, 27]
counts_sum = counts.sum(1, keepdim=True) # [32, 1]
counts_sum_inv = counts_sum**-1 #
probs = counts * counts_sum_inv # [32, 27]
logprobs = probs.log() # [32, 27]
loss = -logprobs[range(n), Yb].mean() 

# pytorch backward pass
for p in parameters:
    p.grad = None
for t in [logprobs, probs, counts_sum_inv, counts_sum, counts, norm_logits, logit_maxes,
          logits, h, z, bn_raw, bn_var_inv, bn_var, bn_del2, 
          bn_del, bn_mean, z_pre, emb, emb_uncat]:
    t.retain_grad()
loss.backward()
loss

---

## Exercise 1 — backpropagate the whole thing, one node at a time

The goal is to reproduce `loss.backward()` by hand. We walk the graph in reverse, and at each node apply the chain rule: the gradient arriving from downstream, times the local derivative of the current operation.

### The guiding principle

Before any matrices, the whole method reduces to one idea, which is cleanest in the scalar case $y = mx$: the derivative $\partial y / \partial x = m$ is "how much a nudge to $x$ moves $y$." Everything that follows is that same statement, generalized to the situation where one input affects the loss through *many* paths at once. When that happens the chain rule says the contributions **add**:

$$
\frac{\partial \mathcal L}{\partial x} \;=\; \sum_{\text{routes } r} \frac{\partial \mathcal L}{\partial (\text{output}_r)} \cdot \frac{\partial (\text{output}_r)}{\partial x}.
$$

I'll write $\partial x \equiv \partial \mathcal L / \partial x$ throughout for the gradient of the loss with respect to $x$. The upstream gradient at each step is whatever we computed one node earlier.

### The first stretch: cross-entropy, node by node

The top of the graph is the cross-entropy computation. Each of these is a short exercise in the rule above:

- **`logprobs`** — the loss is `-logprobs[range(n), Yb].mean()`, so only the entries at the correct labels matter, each with local derivative $-1/n$; everything else is zero.
- **`probs`** — through `logprobs = probs.log()`, local derivative $1/\text{probs}$.
- **`counts_sum_inv`** — `probs = counts * counts_sum_inv` is a broadcast multiply; `counts_sum_inv` is shape `[n,1]` but fans out across all 27 columns, so its gradient sums along the row.
- **`counts`** — this is the first genuinely *multivariable* node: `counts` feeds the loss through **two** routes, once directly via `probs` and once through `counts_sum`. The two contributions add.
- down through **`norm_logits`**, **`logit_maxes`**, and finally **`logits`**.

The one subtlety worth flagging is the max-subtraction. `logit_maxes` is subtracted for numerical stability and, as we'll prove in Exercise 2, contributes *nothing* to the final gradient — the routes through it cancel. The hand-code carries it anyway, and `cmp` confirms it comes out to zero.


In [ ]:
# exercise 1: manually compute the derivative of every variable with respect to loss, check against pytorch backward()

dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0/n
cmp('logprobs', dlogprobs, logprobs)

dprobs = dlogprobs/probs
cmp('dprobs', dprobs, probs)

dcounts_sum_inv = (dprobs*counts).sum(1, keepdim=True)
cmp('dcounts_sum_inv', dcounts_sum_inv, counts_sum_inv)

dcounts_sum = -dcounts_sum_inv*counts_sum**-2
cmp('dcounts_sum', dcounts_sum, counts_sum)

dcounts = dprobs*counts_sum_inv
dcounts += torch.ones_like(counts)*dcounts_sum
cmp('dcounts', dcounts, counts)

dnorm_logits = dcounts * counts
cmp('dnorm_logits', dnorm_logits, norm_logits)

dlogit_maxes = -dnorm_logits.sum(1,keepdim=True)
cmp('dlogit_maxes', dlogit_maxes, logit_maxes)

dlogits = torch.zeros_like(logits)
dlogits[range(logits.shape[0]), logits.max(1).indices] = 1.0
dlogits *= dlogit_maxes
dlogits += dnorm_logits
cmp('dlogits', dlogits, logits)

### Backprop through a linear layer: $\text{logits} = h\,W_2 + b_2$

This is the node that teaches the core move of matrix backprop, so it is worth doing in complete, pedantic detail on a small example before trusting the matrix form. Take $h \in \mathbb{R}^{2\times 5}$, $W \in \mathbb{R}^{5\times 3}$, $b \in \mathbb{R}^{1\times 3}$, giving $l = hW + b \in \mathbb{R}^{2\times 3}$.

The scalar question, asked one entry at a time: *if I nudge $h_{ij}$, which entries of the logits move, and by how much?* Working the perturbation by hand, $h_{ij}$ influences only **row $i$** of the logits (rows don't mix in a matmul), and within that row it is carried by **row $j$ of $W$**. Since it reaches the loss through all three entries of that row, the chain rule sums over them.

#### Gradient with respect to $h$

$$
\begin{aligned}
\partial h_{11} &= \partial l_{11}W_{11} + \partial l_{12}W_{12} + \partial l_{13}W_{13}\\
\partial h_{12} &= \partial l_{11}W_{21} + \partial l_{12}W_{22} + \partial l_{13}W_{23}\\
\partial h_{13} &= \partial l_{11}W_{31} + \partial l_{12}W_{32} + \partial l_{13}W_{33}\\
\partial h_{14} &= \partial l_{11}W_{41} + \partial l_{12}W_{42} + \partial l_{13}W_{43}\\
\partial h_{15} &= \partial l_{11}W_{51} + \partial l_{12}W_{52} + \partial l_{13}W_{53}\\
\partial h_{21} &= \partial l_{21}W_{11} + \partial l_{22}W_{12} + \partial l_{23}W_{13}\\
\partial h_{22} &= \partial l_{21}W_{21} + \partial l_{22}W_{22} + \partial l_{23}W_{23}\\
\partial h_{23} &= \partial l_{21}W_{31} + \partial l_{22}W_{32} + \partial l_{23}W_{33}\\
\partial h_{24} &= \partial l_{21}W_{41} + \partial l_{22}W_{42} + \partial l_{23}W_{43}\\
\partial h_{25} &= \partial l_{21}W_{51} + \partial l_{22}W_{52} + \partial l_{23}W_{53}
\end{aligned}
$$

In general, $\partial h_{ij} = \sum_{b} \partial l_{ib}\, W_{jb}$. The summed index $b$ sits in the **second** slot of both factors; plain matmul contracts the second slot of the left factor against the first of the right, so $W$ must be transposed:

$$
\boxed{\;\partial h = \partial l \, W^{\top}\;}\qquad [2\times 3]\,[3\times 5] \to [2\times 5].
$$

#### Gradient with respect to $W$

Now $W_{jb}$ reaches the loss through **column $b$** of the logits, once per batch row, carried by **column $j$ of $h$**:

$$
\begin{aligned}
\partial W_{11} &= \partial l_{11}h_{11} + \partial l_{21}h_{21}, &\partial W_{12} &= \partial l_{12}h_{11} + \partial l_{22}h_{21}, &\partial W_{13} &= \partial l_{13}h_{11} + \partial l_{23}h_{21}\\
\partial W_{21} &= \partial l_{11}h_{12} + \partial l_{21}h_{22}, &\partial W_{22} &= \partial l_{12}h_{12} + \partial l_{22}h_{22}, &\partial W_{23} &= \partial l_{13}h_{12} + \partial l_{23}h_{22}\\
&\;\;\vdots & &\;\;\vdots & &\;\;\vdots\\
\partial W_{51} &= \partial l_{11}h_{15} + \partial l_{21}h_{25}, &\partial W_{52} &= \partial l_{12}h_{15} + \partial l_{22}h_{25}, &\partial W_{53} &= \partial l_{13}h_{15} + \partial l_{23}h_{25}
\end{aligned}
$$

In general $\partial W_{jb} = \sum_{i} \partial l_{ib}\, h_{ij}$. This time the summed index $i$ (the batch dimension) sits in the **first** slot of both factors, so the *left* factor $h$ is transposed:

$$
\boxed{\;\partial W = h^{\top}\, \partial l\;}\qquad [5\times 2]\,[2\times 3] \to [5\times 3].
$$

#### Gradient with respect to $b$

The bias is broadcast across every row, so each $b_k$ reaches the loss once per batch row with local derivative $1$:

$$
\boxed{\;\partial b = \sum_i \partial l_{i,:} \;=\; \texttt{dlogits.sum(0)}\;}\qquad [2\times 3] \to [1\times 3].
$$

#### The two lessons

First, the transpose is never a convention to memorize — it is *forced* by which index is being summed. Read the index expression as a sentence ("the contracted index sits in slot two of both, so transpose the right factor") and the operation is determined uniquely.

Second, **forward fan-out becomes backward summation.** Wherever the forward pass copies a value to many places — matmul reuse, or a broadcast bias — the backward pass gathers the gradient back from those places by summing. That single duality reappears at every remaining node in the network.

$$
l = hW + b \quad\Longrightarrow\quad \partial h = \partial l\, W^{\top}, \qquad \partial W = h^{\top} \partial l, \qquad \partial b = \textstyle\sum_i \partial l_{i,:}.
$$


### The rest of the graph

With the linear-layer rule in hand, the remainder of Exercise 1 is a sequence of applications of the same two moves (route-sum and fan-out-becomes-sum):

- **`tanh`** — elementwise, local derivative $1 - h^2$.
- **the batch-norm block** — done here the slow way, node by node (`bn_raw`, `bn_var_inv`, `bn_var`, `bn_del2`, `bn_del`, `bn_mean`, `z_pre`). Two things to watch: `bn_del2 → bn_var` carries Bessel's $1/(n-1)$, *not* $1/n$ — you have to differentiate the code you actually wrote, correction constant and all; and `z_pre` reaches the loss through **two** routes (directly through `bn_del`, and indirectly through `bn_mean`, which is itself a function of every `z_pre` in the column). That second, batch-crossing route is the seed of the whole difficulty in Exercise 3.
- **the second linear layer's inputs**, `dW1`, `db1`, `demb`, via the boxed rules above.
- **the `.view` reversal** — `emb = emb_uncat.view(...)` is a pure reshape, so its backward is just reshaping the gradient back to `emb_uncat`'s shape. No arithmetic.
- **the embedding lookup** `C[Xb]` — the last and subtlest node. A single row of `C` can be read many times across the batch (the same character appearing in many contexts). Reading a value multiple times *is* fan-out, so the gradients **scatter-add** back into the rows of `C`. This is `index_add` territory, and the accumulation (`+=`, not `=`) is the entire point: eleven appearances of the letter "a" must pile their gradients onto row `ctoi['a']`, not overwrite each other.

The cell below computes all of it and checks every gradient against PyTorch.


In [ ]:
dh = dlogits @ torch.transpose(W2,0,1)
dW2 = h.T @ dlogits
db2 = dlogits.sum(0,keepdim=True)

dz = dh * (1.0 - h**2)
dbn_raw = dz * bn_gain
dbn_gain = (dz * bn_raw).sum(0,keepdim=True)
dbn_bias = dz.sum(0,keepdim=True)

dbn_var_inv = (dbn_raw * bn_del).sum(0,keepdim = True)
dbn_var = (-0.5*(bn_var + 1e-5)**-1.5) * dbn_var_inv
dbn_del2 = 1.0/(n-1.0) * dbn_var * torch.ones_like(bn_del2)
dbn_del = (dbn_raw * bn_var_inv) + (2*bn_del*dbn_del2)
dbn_mean = -dbn_del.sum(0,keepdim=True)
dz_pre = dbn_del + (dbn_mean/n * torch.ones_like(z_pre))

dW1 = emb.T @ dz_pre
db1 = dz_pre.sum(0)
demb = dz_pre @ W1.T

demb_uncat = demb.view(demb.shape[0], block_size, n_embedding)

dC = torch.zeros_like(C)
for i in range(batch_size):
    for j in range(block_size):
        dC[Xb[i,j]] += demb_uncat[i,j]

cmp('dh', dh, h)
cmp('dW2', dW2, W2)
cmp('db2', db2, b2)
cmp('dz', dz, z)
cmp('dbn_raw', dbn_raw, bn_raw)
cmp('dbn_gain', dbn_gain, bn_gain)
cmp('dbn_bias', dbn_bias, bn_bias)
cmp('dbn_var_inv', dbn_var_inv, bn_var_inv)
cmp('dbn_var', dbn_var, bn_var)
cmp('dbn_del2', dbn_del2, bn_del2)
cmp('dbn_del', dbn_del, bn_del)
cmp('dbn_mean', dbn_mean, bn_mean)
cmp('dz_pre', dz_pre, z_pre)
cmp('dW1', dW1, W1)
cmp('db1', db1, b1)
cmp('demb', demb, emb)
cmp('demb_uncat', demb_uncat, emb_uncat)
cmp('dC', dC, C)


Every line reads `approximate: True`. Some also read `exact: True`; the ones that don't (anything involving a sum, a broadcast, or a different multiply order than autograd's kernel) differ only in the last bit of floating point, with `maxdiff` around `1e-9` to `1e-11`. That is the expected, correct outcome — not a bug to chase.

At this point the entire backward pass of the network has been reconstructed by hand, from `loss` all the way back to `C`. The next two exercises take the two ugliest blocks and collapse them.


---

## Exercise 2 — cross-entropy loss and gradient in one line

The cross-entropy computation in the forward pass threads through six intermediate tensors: `logit_maxes`, `norm_logits`, `counts`, `counts_sum`, `counts_sum_inv`, `probs`, `logprobs`. Backpropagating through all of them works, but each involves an `exp` or `log` or division — numerically touchy and computationally wasteful. The claim is that the gradient of the loss with respect to the logits collapses to something famous and clean. Deriving it is pure algebra.

### Forward: the loss simplifies to log-sum-exp

Write the loss for a single example $i$ with correct label $y$. Substituting each forward line into the next:

$$
\text{loss}_i = -\ln P_{iy} = -\ln\frac{c_{iy}}{\sum_k c_{ik}} = \ln\Big(\sum_k c_{ik}\Big) - \ln c_{iy},
\qquad c_{ik} = e^{\text{logit}_{ik}}.
$$

The $\log$ turns the quotient into a difference, and where the outer $\log$ meets the inner $\exp$ in the second term they cancel ($\ln e^{\text{logit}_{iy}} = \text{logit}_{iy}$), leaving

$$
\boxed{\;\text{loss}_i = \ln\Big(\sum_k e^{\text{logit}_{ik}}\Big) - \text{logit}_{iy}\;}
$$

The first term is the **log-sum-exp** — an irreducible primitive. There is no identity for $\log$ of a sum; this is where the forward simplification stops, and correctly so.

#### Why the max-subtraction is free

The stability trick subtracts a per-row max $m_i$ from every logit. Substituting $\text{logit}_{ik} \to \text{logit}_{ik} - m_i$ and using $e^{a-b}=e^ae^{-b}$, the constant $e^{-m_i}$ pulls out of the sum, so the log-sum-term contributes $-m_i$, while the label term contributes $+m_i$. They cancel exactly. Subtracting the max leaves the loss (and hence every gradient) unchanged — it is a pure numerical convenience, which is *why* it can be dropped when differentiating.

### Backward: the derivative is the softmax minus a one-hot

Differentiate the boxed loss with respect to a single logit $\text{logit}_{ij}$, in two cases.

The label term $-\text{logit}_{iy}$ only contributes when $j = y$, giving $-1$ there.

For the log-sum term, $\frac{\partial}{\partial \text{logit}_{ij}}\ln\!\big(\sum_k e^{\text{logit}_{ik}}\big) = \frac{1}{\sum_k e^{\text{logit}_{ik}}}\cdot \frac{\partial}{\partial \text{logit}_{ij}}\sum_k e^{\text{logit}_{ik}}$. Inside that sum, only the $k=j$ term depends on $\text{logit}_{ij}$ — every other term is a constant and dies. So the sum collapses to a single survivor $e^{\text{logit}_{ij}}$, and

$$
\frac{\partial}{\partial \text{logit}_{ij}}\ln\Big(\sum_k e^{\text{logit}_{ik}}\Big) = \frac{e^{\text{logit}_{ij}}}{\sum_k e^{\text{logit}_{ik}}} = \text{probs}_{ij}.
$$

That fraction *is* the softmax probability we already computed in the forward pass. Combining both terms:

$$
\boxed{\;\frac{\partial \text{loss}_i}{\partial \text{logit}_{ij}} = \text{probs}_{ij} - \mathbb{1}[j = y_i]\;}
$$

The entire six-tensor backward chain collapses to: **take the probabilities, subtract 1 from the correct-class entry, divide by $n$** (for the mean). Conceptually it is *predicted minus target* — the network is pushed to drain probability from the wrong classes and pile it on the right one, in proportion to how wrong it currently is. The same clean gradient falls out of logistic and linear regression; it is what makes softmax-plus-cross-entropy the natural pairing.


In [ ]:
# exercise 2: calculating loss from logits fast
# slow method:
# 
# # linear layer 2 [32, 27]
# logits = h @ W2 + b2 # output layer
# 
# # cross entropy loss (equivalent to F.cross_entropy(logits, Yb)
# logit_maxes = logits.max(1, keepdim=True).values
# norm_logits = logits - logit_maxes # ensures numerical stability
# counts = norm_logits.exp() # [32, 27]
# counts_sum = counts.sum(1, keepdim=True) # [32, 1]
# counts_sum_inv = counts_sum**-1 #
# probs = counts * counts_sum_inv # [32, 27]
# logprobs = probs.log() # [32, 27]
# loss = -logprobs[range(n), Yb].mean() 

# calculated by algebra:
# Loss = 1/n * -∑(ln(P[y]))
# Loss = 1/n * ∑ ln(∑_k e^l_ik) - l_iy
# and derivative d/dl_i = Pi if i =/= y and Pi - 1.0 if i == y


dlogits = 1.0/n * (probs - F.one_hot(Yb, vocab_size)).float()
cmp('dlogits', dlogits, logits)

One line, `approximate: True`, and it agrees to `~1e-9` with the dlogits computed the long way in Exercise 1 — two entirely different routes landing on the same numbers, which is the proof that the collapse is exact rather than approximate.


---

## Exercise 3 — batch normalization backward in one line

This is the hardest derivation in the project, and the one where the scalar-and-routes method earns its keep. The goal mirrors Exercise 2: replace the seven-node batch-norm backward pass with a single formula that takes the gradient arriving at the block's output and produces the gradient at its input `z_pre` directly.

### What makes it hard: the Jacobian is full

Every node so far had a comfortable structure. In an elementwise operation like `tanh`, output $a$ depends only on input $a$ — nudging one input moves exactly one output. Batch norm breaks this. The mean and variance are computed **down each column** (`.sum(0)`), so they depend on *every* entry in the column, and every normalized output subtracts that shared mean and divides by that shared standard deviation.

The consequence: nudging a single input $z\_pre_i$ moves **every** output in its column, through three distinct channels —

1. **directly**, via its own numerator $z\_pre_i - \mu$;
2. **through the mean** $\mu$, which shifts every output;
3. **through the variance** $\sigma^2$, which rescales every output.

If we lay out the column's input→output sensitivities as a grid $J_{ai} = \partial \hat z_a / \partial z\_pre_i$ — one cell per (output $a$, input $i$) pair — an elementwise op would make this grid **diagonal** (off-diagonal cells zero). Batch norm's grid is **full**: every cell is nonzero, because every input touches every output. That fullness is the entire source of difficulty, and it is why the gradient will end up as a sum over the whole column.

A crucial simplification first: because the statistics are per-column, **the 64 columns are completely independent** — 64 parallel batch-norm operations that happen to be stored side by side. So we can fix a single column, drop the column index, and solve a clean $n$-in / $n$-out problem; whatever we derive applies verbatim to all 64.

### The two atoms: how $\mu$ and $\sigma^2$ move

Fix a column. Let $\mu = \frac1n\sum_a z\_pre_a$ and, with Bessel's correction, $\sigma^2 = \frac{1}{n-1}\sum_a (z\_pre_a - \mu)^2$.

**Mean.** Only the $a=i$ term of the sum depends on $z\_pre_i$:

$$
\frac{\partial \mu}{\partial z\_pre_i} = \frac1n.
$$

**Variance.** Each term depends on $z\_pre_i$ both directly (when $a=i$) and through the shared $\mu$. Differentiating and grouping:

$$
\frac{\partial \sigma^2}{\partial z\_pre_i} = \frac{2}{n-1}\sum_a (z\_pre_a - \mu)\big(\mathbb{1}[a{=}i] - \tfrac1n\big) = \frac{2}{n-1}\Big[(z\_pre_i - \mu) - \tfrac1n\underbrace{\textstyle\sum_a (z\_pre_a - \mu)}_{=\,0}\Big] = \frac{2}{n-1}(z\_pre_i - \mu).
$$

The middle term vanishes because **deviations from the mean sum to zero** — the mean is by definition the balance point. This identity, $\sum_a (z\_pre_a - \mu) = 0$, does real work twice in this derivation. Note the $\frac{2}{n-1}$: differentiate the variance you actually wrote (Bessel), and the correction rides through; the textbook $1/n$ variance would give $\frac{2}{n}$ here.


### One cell of the Jacobian

With both atoms in hand, differentiate a general output $\hat z_a = (z\_pre_a - \mu)(\sigma^2+\epsilon)^{-1/2}$ with respect to a general input $z\_pre_i$ — keeping the two indices distinct is what lets us capture the off-diagonal coupling. By the product rule, using the two atoms:

$$
\frac{\partial \hat z_a}{\partial z\_pre_i}
= \underbrace{\big(\mathbb{1}[a{=}i] - \tfrac1n\big)}_{\text{numerator: direct + mean}}(\sigma^2+\epsilon)^{-1/2}
\;+\; (z\_pre_a - \mu)\underbrace{\big(-\tfrac12\big)(\sigma^2+\epsilon)^{-3/2}\,\tfrac{2}{n-1}(z\_pre_i - \mu)}_{\text{variance route}}.
$$

Pulling a single $(\sigma^2+\epsilon)^{-1/2}$ out of both terms and recognizing $\hat z = (z\_pre-\mu)(\sigma^2+\epsilon)^{-1/2}$ collapses this to a strikingly clean form:

$$
\boxed{\;\frac{\partial \hat z_a}{\partial z\_pre_i} = (\sigma^2+\epsilon)^{-1/2}\Big[\,\mathbb{1}[a{=}i] - \tfrac1n - \tfrac{1}{n-1}\,\hat z_a\,\hat z_i\,\Big]\;}
$$

The three terms in the bracket are precisely the three routes: the Kronecker delta is the **direct** route (only the diagonal), the $-\tfrac1n$ is the **mean** route (every $a$), and the $-\tfrac{1}{n-1}\,\hat z_a\hat z_i$ is the **variance** route (every $a$).

### Assembling the full gradient

The loss gradient at $z\_pre_i$ sums this cell against the arriving gradient over *all* outputs the input touched — which, the grid being full, is the whole column:

$$
\frac{\partial \mathcal L}{\partial z\_pre_i} = \sum_a \frac{\partial \mathcal L}{\partial \hat z_a}\,\frac{\partial \hat z_a}{\partial z\_pre_i}.
$$

Let $d\hat z_a = \partial \mathcal L / \partial \hat z_a$ be the gradient arriving at the normalized values (which is just the block-output gradient times the gain, $d\hat z = \gamma\,\partial z$). Substituting the boxed cell, pulling the $a$-independent $(\sigma^2+\epsilon)^{-1/2}$ out front, and distributing the sum across the three bracket terms:

- the $\mathbb{1}[a{=}i]$ term collapses to the single survivor $d\hat z_i$;
- the $-\tfrac1n$ term becomes $-\tfrac1n\sum_a d\hat z_a$ (sum of incoming gradients);
- the variance term becomes $-\tfrac{\hat z_i}{n-1}\sum_a d\hat z_a\,\hat z_a$ (incoming gradients weighted by the normalized values, scaled by this row's own $\hat z_i$).

Putting it together — and folding the gain $\gamma$ back out front since $d\hat z = \gamma\,\partial z$ — gives the collapsed batch-norm backward pass:

$$
\boxed{\;
\frac{\partial \mathcal L}{\partial z\_pre_i}
= \gamma\,(\sigma^2+\epsilon)^{-1/2}\Big[\,\partial z_i \;-\; \tfrac1n\textstyle\sum_a \partial z_a \;-\; \tfrac{\hat z_i}{n-1}\textstyle\sum_a \partial z_a\,\hat z_a\,\Big]
\;}
$$

The two sums $\sum_a \partial z_a$ and $\sum_a \partial z_a\hat z_a$ are single numbers per column — computed once, reused for every row — while $\partial z_i$ and $\hat z_i$ carry the free row index and stay full arrays. In code that is two `.sum(0)` reductions plus elementwise arithmetic: `bn_gain` is $\gamma$, `bn_var_inv` is $(\sigma^2+\epsilon)^{-1/2}$, `dz` is $\partial z$, and `bn_raw` is $\hat z$.


In [ ]:
# exercise 3: batch normalization shortcut, goal - compute dL/dzp with dL/dz, bypass the whole batch norm layer
# old method:

# z_pre = emb @ W1 + b1 # hidden layer pre-activation
#### skip from here...
# bn_mean = 1/n*z_pre.sum(0, keepdim=True) # sum of each row divided by n: [64, 1]
# bn_del = z_pre - bn_mean # delta between pre-activations and row mean: [32, 64]
# bn_del2 = bn_del**2
# bn_var = 1/(n-1)*(bn_del2).sum(0, keepdim=True) # var = sig^2 = del^2/n, with Bessel's correction /(n-1) instead of n
# bn_var_inv = (bn_var + 1e-5)**-0.5 # small scalar to avoid /0
# bn_raw = bn_del * bn_var_inv # intermediate term for cleanliness
#### ... to here!
# z = bn_gain * bn_raw + bn_bias # pre-activation after batch normalization

dz_pre = bn_gain * bn_var_inv * (dz - (1/n)*dz.sum(0,keepdim=True) - (bn_raw/(n-1))*(dz*bn_raw).sum(0,keepdim=True))
cmp('dz_pre', dz_pre, z_pre)

`approximate: True`, matching the seven-node `dz_pre` from Exercise 1 to `~1e-9`. One line replaces the entire `bn_raw → bn_var_inv → bn_var → bn_del2 → bn_del → bn_mean → z_pre` chain, and never materializes any of those intermediates — faster in both directions, and numerically cleaner because it avoids threading gradients back through the `**-0.5` and the squaring.


---

## What the exercise actually taught

The three collapses — `probs - onehot` for cross-entropy, `hᵀ ∂l` and `∂l Wᵀ` for the linear layers, and the three-term expression for batch norm — are the visible output. But the durable result is a single procedure that generated all of them without guessing:

1. **Think in single scalars.** A gradient is always "how much does one output-number move when I nudge one input-number." Matrices are storage; the calculus is scalar.
2. **Trace the routes.** Ask which downstream quantities the nudged input actually reaches. One route for an elementwise op; a whole column for a batch statistic.
3. **Sum over the routes**, each weighted by the gradient already arriving there. This is the multivariable chain rule, and it is the same "fan-out becomes summation" duality every time.
4. **Read the index pattern back as a matrix operation.** Where the contracted index sits determines the transpose; a matched (un-summed) index means elementwise; a summed index means a reduction.

Two ideas recur at every scale. **Forward fan-out becomes backward summation** — matmul reuse, broadcasting, and repeated embedding lookups are all the same phenomenon, and all reverse into a sum. And **the shape of the Jacobian tells you the shape of the answer**: diagonal Jacobians give trivial elementwise backward passes, while a full Jacobian (batch norm) forces a sum over the coupled dimension. Nothing in the network needed more than this, and nothing in a larger network — softmax attention, convolutions, normalization variants — needs more than this either. The bookkeeping gets denser; the method does not change.
